# Chapter 10: Unsupervised RFM Customer Segmentation & K-Means
### *Modern Machine Learning & Data Science Engineering* by Vaibhav Rastogi

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/v893979/Modern_Machine_Learning_and_Data_Science_Engineering/blob/main/src/ch10_rfm_customer_segmentation.ipynb)

---

## 📌 Notebook Overview
Transform skewed transaction data using log scaling and cluster customers into executive personas using K-Means.

In [ ]:
# Environment & Data Auto-Download for Google Colab
import os
import urllib.request

os.makedirs('../data', exist_ok=True)
data_path = "../data/customer_rfm.csv"
if not os.path.exists(data_path):
    url = "https://raw.githubusercontent.com/v893979/Modern_Machine_Learning_and_Data_Science_Engineering/main/data/customer_rfm.csv"
    print(f"Downloading customer_rfm.csv from GitHub...")
    urllib.request.urlretrieve(url, data_path)
    print("Download complete!")


### 1. Log Transformation & Scaling

Apply $\ln(1 + x)$ to cure extreme financial right-skew:

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/customer_rfm.csv")
display(df.head())

# Log transform & standardize
X = np.column_stack([np.log1p(df["recency_days"]), np.log1p(df["frequency_purchases"]), np.log1p(df["monetary_value"])])
X_scaled = (X - X.mean(axis=0)) / X.std(axis=0)

### 2. K-Means Clustering & Persona Profiling

Iterate cluster centroids until convergence and profile customer segments:

In [ ]:
np.random.seed(42)
centroids = X_scaled[np.random.choice(len(X_scaled), size=4, replace=False)].copy()

for it in range(20):
    dists = np.linalg.norm(X_scaled[:, np.newaxis, :] - centroids[np.newaxis, :, :], axis=2)
    labels = np.argmin(dists, axis=1)
    new_centroids = np.array([X_scaled[labels == k].mean(axis=0) for k in range(4)])
    if np.allclose(centroids, new_centroids, atol=1e-4):
        print(f"Converged in {it+1} iterations!")
        break
    centroids = new_centroids

df["cluster"] = labels
summary = df.groupby("cluster").agg(
    users=("customer_id", "count"),
    avg_recency=("recency_days", "mean"),
    avg_freq=("frequency_purchases", "mean"),
    avg_spend=("monetary_value", "mean")
).reset_index()
display(summary)